In [1]:
import xskillscore as xs
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
from scipy.spatial.distance import cdist
import matplotlib.ticker as ticker

In [10]:
# ---------------------------------------
# paths
# ---------------------------------------
# Elevation data from Vinther et al. (2009)
vinther = xr.open_dataset("../../FesmData/Vinther2009_elevations/vinther2009.nc")

# Ensemble elevations 
yelmo_elev = xr.open_dataset("../output/ensemble_elevations.nc")
yelev = yelmo_elev.sel(time=slice(-11700, -40)) # Years covered by Vinther reconstruction
vint = vinther.sel(time=yelev.time) # Remove years bellow yelmo resolution
sim0 = xr.open_dataset("/home/luciagu/projects/3_yelmo_deglaciation/ensemble_reduced/0/yelmo2D_reduced.nc")

# Ensemble GIA 
ygia=xr.open_dataset("../output/ensemble_gia.nc")

# Ensemble RSL 
yrsl = xr.open_dataset("../output/ensemble_rsl.nc")

# Present-day data
bedm = xr.open_dataset("/p/projects/megarun/ice_data/Greenland/GRL-8KM/GRL-8KM_TOPO-M17.nc")
vel = xr.open_dataset("/p/projects/megarun/ice_data/Greenland/GRL-8KM/GRL-8KM_VEL-J18.nc")
rg = xr.open_dataset("/p/projects/megarun/ice_data/Greenland/GRL-8KM/GRL-8KM_REGIONS.nc")
bedm = bedm.assign_coords(xc=sim0.xc, yc=sim0.yc)
rg = rg.assign_coords(xc=sim0.xc, yc=sim0.yc)
vel = vel.assign_coords(xc=sim0.xc, yc=sim0.yc)
schu = xr.open_dataset("../../FesmData/Schumacher2018_GIA_GrIS/data/schumacher2018_GR.nc")
# Past data
gowan = xr.open_dataset("../../FesmData/GAPSLIP_Gowan2023_GrIS/output/rsl_dataset_reduced.nc")
leger = xr.open_dataset("../../FesmData/Leger2024_PaleoGris/output/paleogris_8km.nc")
leger["xc"] = sim0.xc # They differ in 0.00006km 
leger["yc"] = sim0.yc
# Ensemble
path_ensemble="../../ensemble_reduced/*"
file_name = "yelmo2D_reduced.nc" 
sim_paths = sorted(glob.glob(path_ensemble))



In [ ]:
# Scores calculation
# For the present-day observables:
# sc_pd1 --> H_ice rmse (Bedmachine)                                  X
# sc_pd2 --> ice cover (Bedmachine)                                   X
# sc_pd3 --> z_bed rmse (Bedmachine)                                  X
# sc_pd4 --> uxy_s rmse                                               X
# sc_pd5 --> uplift rate chi square (Schumacher et al. 2018)          X
# --> For all sc_pd we can do regional scores as well                 PENDIENTE

# For past observables:
# sc_pt1n --> NGRIP elevations Normalized chi2 (Vinther et al. 2009)  X
# sc_pt1g --> GRIP elevations Normalized chi2                         X   
# sc_pt1c --> Camp Century elevations Normalized chi2                 X
# sc_pt1d --> DYE3 elevations Normalized chi2                         X
# sc_pt2 --> isocrhones chi2 (Leger et al. 2024)                      X
# sc_pt3 --> LGM extension                                            PENDIENTE
# sc_pt4 --> RSL chi2 (Gowan 2023)                                    X
# --> For sc_pt 2-4 we can do regional scores as well                 PENDIENTE

In [117]:
# Past scores - elevations
def score_elev(yelmo,ds,icec):
    ens_elev= yelmo.sel(ice_core=icec).z_srf
    obs_elev= ds.sel(ice_core=icec).z_srf
    sigma=ds.sel(ice_core=icec).error
    chi2 = ((ens_elev - obs_elev)**2 / sigma**2).sum(dim="time")
    chi2_red = chi2 / len(obs_elev.time)
    return chi2_red.values

sc_pt1g=score_elev(yelev,vint,"grip")
sc_pt1n=score_elev(yelev,vint,"ngrip")
sc_pt1c=score_elev(yelev,vint,"camp_century")
sc_pt1d=score_elev(yelev,vint,"dye3")

In [ ]:
#Past scores - RSL 
#  Interpolate simulations in time 
yrsl_interp = yrsl.interp(time=np.arange(yrsl.time.min(), yrsl.time.max(), 20), method="linear")

delta_rsl_list = []
for site in gowan.region.values:
    obs_site = gowan.sel(region=site)
    delta_rsl_site = []
    for i,t in enumerate(obs_site.time[~np.isnan(obs_site.time.values)]):
        t_min = obs_site.time[i] - obs_site.time_err[i]
        t_max = obs_site.time[i] + obs_site.time_err[i]

        model_in_window = yrsl_interp.sel(region=site).where((-yrsl_interp.time >= t_min) & (-yrsl_interp.time <= t_max), drop=True)
        if model_in_window.rsl.size == 0:
            print("error: site ", site)
            print("tmin=",t_min.values)
            print("t_max=",t_max.values)
        else:
            sigma=obs_site.RSL_err_max[i]+obs_site.RSL_err_min[i]
            diffs = (model_in_window.rsl - obs_site.RSL[i])/(sigma)
            idx_min = np.abs(diffs).argmin(dim='time')
            diff = diffs.isel(time=idx_min)     
        
            delta_rsl_site.append(diff.values)
    diff_rsl = np.stack(delta_rsl_site, axis=0)
    chi2 = (diff_rsl**2).sum(axis=0)
    chi2_red = chi2 / len(diff_rsl)
    delta_rsl_list.append(chi2_red)

# Array regions x sims (47, 3000)
ch2_complete = np.stack(delta_rsl_list, axis=0)

# One score for all the regions, for later keep one score per basin
sc_pt4 = ch2_complete.mean(axis=0)


error: site  Kap_Clarence_Wyckoff
tmin= 31627.0
t_max= 33319.0
error: site  Kap_Clarence_Wyckoff
tmin= 26979.0
t_max= 27751.0
error: site  Kap_Clarence_Wyckoff
tmin= 33296.0
t_max= 34568.0
error: site  Kap_Clarence_Wyckoff
tmin= 41022.0
t_max= 43864.0
error: site  Kap_Morris_Jesup
tmin= 42578.0
t_max= 44958.0
error: site  Kap_Morris_Jesup
tmin= 36522.0
t_max= 39388.0
error: site  Kap_Morris_Jesup
tmin= 39176.0
t_max= 40910.0
error: site  Kap_Morris_Jesup
tmin= 40101.0
t_max= 47643.0
error: site  Nansen_land
tmin= 40699.0
t_max= 41191.0


In [118]:
# Present-day gia scores TERMINADO
chi2_stations = []
for sta in ygia.station:
    ygia_sel = ygia.sel(station=sta)
    schu_sel = schu.sel(station=sta)
    chi2 =((ygia_sel.v_vert.values - schu_sel.v_vert.values)/schu_sel["std"].values)**2
    chi2_stations.append(np.sqrt(chi2))

chi2 = np.stack(chi2_stations, axis=0)
sc_pd5 = np.mean(chi2,axis=0)

In [119]:
# Present-day topographic scores TERMINADO

rmse_ens_H = []
rmse_ens_vel = []
rmse_ens_area = []
rmse_ens_z = []
valid_sim_indices = []   

for i, sim_path in enumerate(sim_paths):
    file_path = os.path.join(sim_path, file_name)
    n_sim = int(os.path.basename(sim_path))
    try:
        yelmo = xr.open_dataset(file_path)
    except FileNotFoundError:
        print(f"[ERROR] No se encontró el archivo: {file_path}. Se omite esta simulación.")
        continue
    
    sim=yelmo.sel(time=0)
    
    # sc_pd1 H_ice
    diff = (sim.H_ice.values - bedm.H_ice.values)**2
    rmse_H = np.sqrt(diff.mean()) # mean in x and y
    rmse_ens_H.append(rmse_H)
    
    # sc_pd2 ice cover
    mask_yelmo = xr.where((sim.H_ice > 0)&(rg.mask==1.3), 1, 0)
    mask_bedm = xr.where((bedm.H_ice > 0)&(rg.mask==1.3), 1, 0)
    diff = (mask_yelmo.values - mask_bedm.values)**2
    rmse_area = np.sqrt(diff.mean()) # mean in x and y
    rmse_ens_area.append(rmse_area)
    
    # sc_pd3 z_bed
    diff = (sim.z_bed.values - bedm.z_bed.values)**2
    rmse_z = np.sqrt(diff.mean()) # mean in x and y
    rmse_ens_z.append(rmse_z)
    
    # sc_pd4 uxy_s
    diff1 = (sim.uxy_s - vel.uxy_srf)**2
    diff = xr.where(diff1==np.nan, 0, diff1) #outside the domain where there is no ice the data is nan, so we impose 0
    rmse_vel = np.sqrt(diff.mean()) # mean in x and y
    rmse_ens_vel.append(rmse_vel)
    
    valid_sim_indices.append(n_sim)   

sc_pd1 = np.stack(rmse_ens_H, axis=0)
sc_pd2 = np.stack(rmse_ens_area, axis=0)
sc_pd3 = np.stack(rmse_ens_z, axis=0)
sc_pd4 = np.stack(rmse_ens_vel, axis=0)
valid_sim_indices = np.stack(valid_sim_indices, axis=0)


In [ ]:
# TERMINADO
ds_ensemble = xr.Dataset(
    data_vars={
        "sc_pd1": (("sim"), sc_pd1),
        "sc_pd2": (("sim"), sc_pd2),
        "sc_pd3": (("sim"), sc_pd3),
        "sc_pd4": (("sim"), sc_pd4)},
    coords={
        "sim": np.array(valid_sim_indices)})
scores1 = ds_ensemble.sortby("sim")
scores1["sc_pd5"]= xr.DataArray(data=sc_pd5, coords={"sim": ygia.sim.values}, dims=("sim"))
scores1["sc_pt1g"]= xr.DataArray(data=sc_pt1g, coords={"sim": yelev.sim.values}, dims=("sim"))
scores1["sc_pt1n"]= xr.DataArray(data=sc_pt1n, coords={"sim": yelev.sim.values}, dims=("sim"))
scores1["sc_pt1c"]= xr.DataArray(data=sc_pt1c, coords={"sim": yelev.sim.values}, dims=("sim"))
scores1["sc_pt1d"]= xr.DataArray(data=sc_pt1d, coords={"sim": yelev.sim.values}, dims=("sim"))
scores1["sc_pt4"]= xr.DataArray(data=sc_pt4, coords={"sim": yrsl.sim.values}, dims=("sim"))

scores1.to_netcdf("/home/luciagu/projects/3_yelmo_deglaciation/GrIS_deglaciation_yelmox/scoring/scores/sc_all.nc")

In [ ]:
# Past scores - isochrones

chi2_ens = []
valid_sim_indices = []     
sigma=xr.where(leger.err==0, np.nan, leger.err*1e-3)

for path in sim_paths:
    full_path = f"{path}/{file_name}"
    n_sim = int(os.path.basename(path))
    try:
        with xr.open_dataset(full_path) as sim:
            diff = leger.age*1e-3 - sim.isochrone.where(leger.age*1e-3<14)
            chi2 = (diff**2) / (sigma**2)
            
            chi2_values = np.array(chi2).flatten()
            chi2_clean = chi2_values[~np.isnan(chi2_values)]
            chi_mean = np.mean(chi2_clean)

            chi2_ens.append(chi_mean)
            valid_sim_indices.append(n_sim)   
    except FileNotFoundError:
        print(f"Archivo no encontrado en: {full_path}")

chi2_ens = np.array(chi2_ens)
valid_sim_indices = np.array(valid_sim_indices)

ds_isoc = xr.Dataset(
    data_vars={
        "sc_pt2": (("sim"), chi2_ens)},
    coords={
        "sim": np.array(valid_sim_indices)})
scores_isoc = ds_isoc.sortby("sim")

scores2=xr.open_dataset("/home/luciagu/projects/3_yelmo_deglaciation/GrIS_deglaciation_yelmox/scoring/scores/sc_all.nc")
scores_final = xr.merge([scores_isoc, scores2])

scores_final.to_netcdf("./scores/sc_all2.nc")


In [75]:
weights = {
    "sc_pt2": 0.2,
    "sc_pd1": 0.2,
    "sc_pd2": 0.2,
    "sc_pd3": 0.2,
    "sc_pd4": 0.2,
    "sc_pd5": 1,
    "sc_pt1g": 0.2,
    "sc_pt1n": 0.2,
    "sc_pt1c": 0.2,
    "sc_pt1d": 0.2,
    "sc_pt4": 0.2,
}

def normalize(da):
    # (valor - minimo) / (maximo - minimo)
    return (da - da.min()) / (da.max() - da.min())

weighted_total = sum(normalize(scores_final[k]) * weights[k] for k in weights) / sum(weights.values())

best_idx = weighted_total.argmin(dim='sim')
best_idx

<xarray.DataArray ()> Size: 8B
array(101)